# Tavily 预定义搜索工具

参考 [《LangChain-Tool调用》第 5.3 节](../LangChain-Tool调用.md)。项目已声明所需依赖，初次使用可在项目根目录运行 `uv sync`，然后选择项目的 `.venv` 作为 Notebook 内核。

在项目根目录的 `.env` 中配置 `TAVILY_API_KEY`。运行 DeepSeek Agent 示例时，再配置 `DEEPSEEK_API_KEY`。

## 1. 直接调用 Tavily

`TavilySearch` 是预定义工具，实例化后可以直接用 `invoke()` 搜索。`max_results` 控制返回条数，`topic="general"` 表示通用网页搜索。

In [1]:
from dotenv import find_dotenv, load_dotenv
from langchain_tavily import TavilySearch

# 从当前工作目录向上查找 .env
load_dotenv(find_dotenv(usecwd=True))

tavily = TavilySearch(max_results=5, topic="general")

In [2]:
search_result = tavily.invoke({"query": "杭州今天天气如何？"})

if isinstance(search_result, dict):
    for item in search_result["results"]:
        print(item["title"])
        print(item["url"])
        print(item["content"][:200], end="\n\n")
else:
    print(search_result)

今日天气丨杭州今天入夏 浙江气温逐渐升高--新蓝网
https://www.cztv.com/newsDetail/787136
天天说天气：入夏啦！大片30度

杭州市气象台发布消息：从今天（5月12日）开始，杭州连续五天平均气温高于22℃，进入气象学意义上的夏季。

此前，台州、丽水、金华已经进入气象学意义上的夏季。

杭州常年入夏时间5月16日，最早入夏4月29日（1994年），最晚入夏6月8日（1985年），在杭州，夏季待机时间几乎是四季中最长的。

入夏之后天气晴雨相间，气温逐渐升高。今明两天（12-13日）仍是晴

【杭州天气预报】杭州天气预报7天,10天,15天_全国天气网
https://tianqi.so.com/weather/101210101
6

空气 良西南风 2级

- 今天 (11-22)晴8/17℃东北风 微风

<!--THE END-->

- 明天 (11-23)多云9/17℃西南风 微风

<!--THE END-->

- 周二 (11-24)多云转晴5/13℃东北风 微风

<!--THE END-->

- 周三 (11-25)多云7/16℃东北风 微风

<!--THE END-->

- 周四 (11-26)

杭州, 浙江省, 中國 三日天氣預報 | AccuWeather
https://www.accuweather.com/zh/cn/hangzhou/106832/weather-forecast/106832
## 目前天氣

AM12:49

6°C

RealFeel® 4°

晴朗 更多詳情

風 西北偏西 每小时11公里

風速 每小时14公里

空氣品質 不佳

## 展望未來

天氣宜人 週日

## 每小時預測

## 太陽和月亮

10 小時 33 分鐘

日出 AM6:28

日落 PM5:01

12 小時 10 分鐘

日出 PM1:24

日落 AM1:34

## 空氣品質



杭州天气预报,杭州7天天气预报,杭州15天天气预报,杭州天气查询
https://www.weather.com.cn/weather/101210101.shtml
首页 预报 预警 雷达 云图 天气地图 专业产品 资讯 视频 节气

更多

台风路径 空间天气 图片 专题 环境 旅游 碳中和 气象

## 2. 让 Agent 调用 Tavily

把同一个 `tavily` 实例传给 `create_agent()`。使用 `stream_mode="updates"` 查看模型发出工具调用、工具返回结果和模型生成答案的过程。

In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}},
)
agent = create_agent(
    model=model,
    tools=[tavily],
    system_prompt="涉及实时信息时先使用网页搜索，并在回答中注明来源。",
)

for update in agent.stream(
    {"messages": [HumanMessage(content="请搜索北京未来五天的天气预报。")]},
    stream_mode="updates",
):
    for step, state in update.items():
        print(f"步骤：{step}")
        for message in state["messages"]:
            if isinstance(message, ToolMessage):
                print(f"{message.name} 返回：{str(message.content)[:300]}")
            else:
                message.pretty_print()

步骤：model
================================== Ai Message ==================================

我来帮您搜索北京未来五天的天气预报。
Tool Calls:
  tavily_search (call_00_ZJAAmgF2wH3giqV3n2db7590)
 Call ID: call_00_ZJAAmgF2wH3giqV3n2db7590
  Args:
    query: 北京未来五天天气预报
    topic: general
    search_depth: advanced
    time_range: week
  tavily_search (call_01_odF6SnhWaaK3mKrwdtzJ0736)
 Call ID: call_01_odF6SnhWaaK3mKrwdtzJ0736
  Args:
    query: 北京天气预报 未来一周 气温
    topic: news
    search_depth: advanced
    time_range: week
步骤：tools
tavily_search 返回：{"query": "北京未来五天天气预报", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://qq.ip138.com/weather/beijing/dongchengqu_wfj_15tian.htm", "title": "王府井天气预报,北京王府井未来15天天气预报,北京王府井天气预报", "content": "5天天气预报\n 7天天气预报\n 一周天气预报\n 10天天气预报\n 两周天气预报\n 15天天气预报\n\n|  |  |  |  |  |
步骤：tools
tavily_search 返回：{"query": "北京天气预报 未来一周 气温", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.weather.com.cn/html/we